# Etapa 2 (Análise)

In [1]:
# criando seção spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("INPE-MLlib")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

d:\data_science\inpe-mllib-study\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Após executar o notebook `1_data_preparation.ipynb`, vamos ler os dados que foram salvos para começar a parte da análise do modelo

In [2]:
df = spark.read.parquet(
    "../data/bronze/inpe_inmet/"
)

Documentando o dataset:

In [4]:
df.printSchema()

root
 |-- id_foco: long (nullable = true)
 |-- data_hora_foco: timestamp (nullable = true)
 |-- satelite: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- bioma: string (nullable = true)
 |-- frp: double (nullable = true)
 |-- latitude_foco: double (nullable = true)
 |-- longitude_foco: double (nullable = true)
 |-- codigo_estacao: string (nullable = true)
 |-- nome_estacao: string (nullable = true)
 |-- latitude_estacao: double (nullable = true)
 |-- longitude_estacao: double (nullable = true)
 |-- distancia_estacao_km: double (nullable = true)
 |-- temperatura_ar_c: double (nullable = true)
 |-- umidade_relativa_ar_pct: double (nullable = true)
 |-- precipitacao_total_horario_mm: double (nullable = true)
 |-- vento_velocidade_horaria_ms: double (nullable = true)
 |-- radiacao_global_kj_m2: double (nullable = true)
 |-- ano: integer (nullable = true)



Temos 20 colunas e os tipos estão corretos porque tratamos antes na parte de preparação dos dados

In [3]:
df.count()

8951835

Temos 8951835 registros

In [4]:
df.describe().show()

+-------+--------------------+---------+---------+-------------+--------------+------------------+-----------------+------------------+-------------------+--------------+------------+------------------+------------------+--------------------+------------------+-----------------------+--------------------+---------------------+--------------------------------+---------------------------+---------------------+------------------+
|summary|             id_foco| satelite|   estado|    municipio|         bioma|               frp|    dia_sem_chuva|     latitude_foco|     longitude_foco|codigo_estacao|nome_estacao|  latitude_estacao| longitude_estacao|distancia_estacao_km|  temperatura_ar_c|umidade_relativa_ar_pct|precipitacao_inpe_mm|precipitacao_inmet_mm|pressao_atmosferica_nivel_mar_mb|vento_velocidade_horaria_ms|radiacao_global_kj_m2|               ano|
+-------+--------------------+---------+---------+-------------+--------------+------------------+-----------------+------------------+---

In [5]:
df.show()

+------------+-------------------+--------+----------+--------------------+--------------+-----+-------------+------------------+------------------+--------------+----------------+----------------+-----------------+--------------------+----------------+-----------------------+--------------------+---------------------+--------------------------------+---------------------------+---------------------+----+
|     id_foco|     data_hora_foco|satelite|    estado|           municipio|         bioma|  frp|dia_sem_chuva|     latitude_foco|    longitude_foco|codigo_estacao|    nome_estacao|latitude_estacao|longitude_estacao|distancia_estacao_km|temperatura_ar_c|umidade_relativa_ar_pct|precipitacao_inpe_mm|precipitacao_inmet_mm|pressao_atmosferica_nivel_mar_mb|vento_velocidade_horaria_ms|radiacao_global_kj_m2| ano|
+------------+-------------------+--------+----------+--------------------+--------------+-----+-------------+------------------+------------------+--------------+----------------+--

Interessante é perceber que em relação a variável alvo, a média é 33, enquanto o valor máximo é 9612, provavelmente temos poucos casos com esse valor máximo. Outra coisa interessante é que a precipitação média é quase 0, ou seja, tem muito mais horas sem chover do que chovendo, o que é esperado. O valor da velocidade do vento tem média 1 com dp de 1.5, enquanto o valor máximo é 13, mostrando mais uma vez essa discrepância, já a radiação global parece mais correta.

Analisando a distribuição da variável alvo (FRP):

In [3]:
# vamos criar a variável alvo (1 se maior que a mediana de frp e vice-versa)
# primeiro pegando a mediana (quantil 0.5)
from pyspark.sql import functions as F

mediana_frp = df.approxQuantile("FRP", [0.5], 0.001)[0]

print(mediana_frp)

9.0


In [4]:
# agora criando a variável binária para analisar as classes

df = df.withColumn(
    'target',
    F.when(F.col('FRP') > mediana_frp, 1).otherwise(0)
)

df.select('target').show(5)

+------+
|target|
+------+
|     0|
|     0|
|     0|
|     1|
|     1|
+------+
only showing top 5 rows


Verificando a proporção de cada classe:

In [5]:
total = df.count()

distribuicao = (
    df.groupBy('target')
    .count()
    .withColumn(
        "proporcao",
        F.round(F.col("count")/F.lit(total), 3)
    )
)

distribuicao.show()

+------+-------+---------+
|target|  count|proporcao|
+------+-------+---------+
|     1|3655499|    0.408|
|     0|5296336|    0.592|
+------+-------+---------+



Percebemos que temos muito mais valores abaixo da mediana (target = 0) com aproximadamente 59,2% enquanto temos menos valores acima da mediana (target = 1), com 40,8%, o que é esperado considerando que temos mais focos de incendio menores do que maiores.

Vamos dropar a target por enquanto e depois voltamos com ela definitivamente

In [6]:
df = df.drop('target')

Analisando o percentual de valores nulos por coluna:

In [7]:
total = df.count()

df.select([
    F.round(
        F.sum(F.col(c).isNull().cast("int")) / F.lit(total) * 100,
        2
    ).alias(c)
    for c in df.columns
]).show()

+-------+--------------+--------+------+---------+-----+-----+-------------+-------------+--------------+--------------+------------+----------------+-----------------+--------------------+----------------+-----------------------+--------------------+---------------------+--------------------------------+---------------------------+---------------------+---+
|id_foco|data_hora_foco|satelite|estado|municipio|bioma|  frp|dia_sem_chuva|latitude_foco|longitude_foco|codigo_estacao|nome_estacao|latitude_estacao|longitude_estacao|distancia_estacao_km|temperatura_ar_c|umidade_relativa_ar_pct|precipitacao_inpe_mm|precipitacao_inmet_mm|pressao_atmosferica_nivel_mar_mb|vento_velocidade_horaria_ms|radiacao_global_kj_m2|ano|
+-------+--------------+--------+------+---------+-----+-----+-------------+-------------+--------------+--------------+------------+----------------+-----------------+--------------------+----------------+-----------------------+--------------------+---------------------+-----

Primeiramente, olhando para a nossa variável alvo, o FRP, temos aproximadamente 18% dos dados faltantes. Isso é um ponto importante, principalmente porque usamos o FRP para criar a nossa target binária, onde valores acima da mediana recebem 1 e valores abaixo ou iguais recebem 0. O problema é que valores nulos podem acabar sendo classificados como 0 mesmo sem sabermos o valor real do FRP. Por isso, precisamos tratar esses casos antes de calcular a mediana e criar a target. Como o FRP é justamente a variável que queremos prever, acredito que o mais adequado seja remover esses registros, ao invés de tentar preencher os valores faltantes com média ou mediana.

Olhando agora para as variáveis relacionadas à precipitação e umidade, que são muito importantes para a nossa hipótese principal, temos aproximadamente 27,5% de valores faltantes para umidade relativa. Para precipitação, temos duas fontes diferentes: a precipitação fornecida pelo INPE, com apenas 2,78% de valores faltantes, e a precipitação proveniente das estações do INMET, com aproximadamente 31,46%. Como os dados do INPE possuem uma cobertura consideravelmente maior, essa será utilizada como a principal fonte de precipitação. Entretanto, nos casos em que o valor do INPE estiver ausente e existir uma medição disponível pelo INMET, podemos utilizar esse valor como alternativa através de um `coalesce`. Dessa forma, priorizamos valores realmente observados antes de recorrer à imputação. Os casos em que nenhuma das duas fontes possuir informação serão tratados posteriormente na etapa específica de imputação de valores nulos.

Também temos a variável `dia_sem_chuva`, que representa a quantidade de dias consecutivos sem ocorrência de chuva e apresenta aproximadamente 4,42% de valores faltantes. Essa variável é interessante para a hipótese principal, pois complementa a informação de precipitação ao representar a duração da condição seca antes da ocorrência do foco. Como seu percentual de valores ausentes é relativamente baixo, esses casos também poderão ser tratados posteriormente por meio de imputação, preservando a maior parte dos registros.

Para a primeira hipótese secundária, o problema acaba sendo parecido, já que ela depende principalmente da precipitação e da umidade para identificar regiões mais secas e verificar se existe uma maior concentração de focos com target = 1 nessas áreas. Portanto, a forma como esses valores faltantes forem tratados pode afetar diretamente essa análise, sendo importante definir o tratamento antes de realizar a parte espacial.

Já para a segunda hipótese secundária temos uma limitação bem maior, pois aproximadamente 86,1% dos valores de velocidade do vento estão faltando. Nesse caso, preencher os dados usando média ou mediana provavelmente não seria uma boa opção, já que estaríamos estimando artificialmente a maior parte da coluna. Apesar do percentual elevado, a quantidade absoluta de registros disponíveis ainda pode ser suficiente para realizar essa análise. Dessa forma, a hipótese poderá ser analisada separadamente utilizando apenas o subconjunto de registros que possuem informação de vento, verificando também se esses dados estão concentrados em determinadas estações ou períodos.

Por fim, temperatura do ar, pressão atmosférica e radiação global possuem aproximadamente 92,7%, 92,5% e 93,7% de valores faltantes, respectivamente. Como essas variáveis não são centrais para as hipóteses propostas e possuem poucos dados disponíveis, não parece adequado preencher mais de 90% de seus valores artificialmente. Também foi considerada a possibilidade de utilizá-las em um modelo complementar apenas com os registros completos, porém a interseção entre essas variáveis resultou em uma quantidade muito pequena de observações. Por esse motivo, a decisão mais adequada é removê-las das análises e dos modelos.


Falando dos dados em si, temos uma granularidade de um foco com uma estação mais próxima associada, tendo informações de FRP (Fire Radiative Power), precipitação, umidade e etc, e assim podemos trabalhar com isso para testar as hipóteses e construir o modelo de classificação

# Etapa 3

Tarefa: Filtrar registros inválidos ou irrelevantes, documentando cada filtro e quantos registros foram removidos

A primeira coisa que chama atenção aqui é o FRP negativo, isso é inválido e pode ser considerado um filtro, vamos retirar esses valores

In [10]:
df.count()

8951835

In [8]:
# aplicando o filtro de frp < 0

df = df.filter(
    F.col("frp").isNull() | (F.col("frp") >= 0)
)

In [12]:
df.count()

8951830

Houve uma diminuição de 5 registros

Agora como a análise é para o Nordeste, vamos verificar quanto aos estados se está tudo certo

In [9]:
df.groupBy("estado").count().orderBy("estado").show()

+-------------------+-------+
|             estado|  count|
+-------------------+-------+
|            ALAGOAS|  44542|
|              BAHIA|1264509|
|              CEARÁ| 383202|
|   DISTRITO FEDERAL|   2033|
|     ESPÍRITO SANTO|  50194|
|              GOIÁS| 243196|
|           MARANHÃO|2366934|
|        MATO GROSSO|  13754|
|       MINAS GERAIS| 335596|
|            PARAÍBA|  97999|
|               PARÁ|1303467|
|         PERNAMBUCO| 141339|
|              PIAUÍ|1351724|
|RIO GRANDE DO NORTE|  63480|
|            SERGIPE|  20975|
|          TOCANTINS|1268886|
+-------------------+-------+



Temos alguns estados que não são do Nordeste aqui, vamos filtrar

In [10]:
from pyspark.sql import functions as F

estados_nordeste = [
    "ALAGOAS",
    "BAHIA",
    "CEARÁ",
    "MARANHÃO",
    "PARAÍBA",
    "PERNAMBUCO",
    "PIAUÍ",
    "RIO GRANDE DO NORTE",
    "SERGIPE"
]

df = df.filter(
    F.col("estado").isin(estados_nordeste)
)

In [15]:
df.count()

5734704

Houve uma diminuição significativa aqui de 3.217.126 registros, mas agora temos realmente apenas focos do nordeste, ficando com 5734704

Outra análise que pode ser feita é em relação a distância, porque quanto maior for a distância, menor é a influência daquela condição meteorológica sobre aquele foco

In [16]:
df.approxQuantile(
    "distancia_estacao_km",
    [0.25, 0.50, 0.75, 0.90, 0.95, 0.96, 0.99],
    0.001
)

[34.89239129842546,
 52.305688069481725,
 70.65230980317692,
 86.24898313526768,
 96.59247043662153,
 100.2186232738692,
 137.27469775510528]

In [17]:
df.select(
    F.sum((F.col("distancia_estacao_km") <= 50).cast("int"))
        .alias("ate_50km"),

    F.sum((F.col("distancia_estacao_km") <= 100).cast("int"))
        .alias("ate_100km"),

    F.sum((F.col("distancia_estacao_km") <= 150).cast("int"))
        .alias("ate_150km"),

    F.sum((F.col("distancia_estacao_km") <= 200).cast("int"))
        .alias("ate_200km")
).show()

+--------+---------+---------+---------+
|ate_50km|ate_100km|ate_150km|ate_200km|
+--------+---------+---------+---------+
| 2679743|  5502156|  5688096|  5727153|
+--------+---------+---------+---------+



In [18]:
from pyspark.sql import functions as F

df.select(
    F.count("*").alias("total"),
    
    F.sum(
        (F.col("distancia_estacao_km") <= 100).cast("int")
    ).alias("ate_100km"),
    
    F.sum(
        (F.col("distancia_estacao_km") > 100).cast("int")
    ).alias("acima_100km")
).show()

+-------+---------+-----------+
|  total|ate_100km|acima_100km|
+-------+---------+-----------+
|5734704|  5502156|     232548|
+-------+---------+-----------+



In [11]:
df = df.filter(
    F.col("distancia_estacao_km") <= 100
)

O interessante aqui é perceber que até 100km temos 96% dos dados, o que é interessante considerando preservar a representabilidade dos efeitos meteorológicos, então podemos aplicar um corte de até 100km para melhorar nossos testes sem ter uma perda significativa, que nesse caso será de 4%, ou então 232548 registros, ficando com 5502156 registros de até 100km

In [21]:
df.count()

5502156

Com esses filtros agora temos um dataset mais válido e podemos partir para a próxima parte de valores nulos:

Tratar valores nulos: escolher entre exclusão, imputação (média/mediana/moda) ou flag. Justifique a escolha para cada coluna.

A primeira variável a ser analisada é a frp, considerando nossa análise anterior, vamos apenas ver novamente quantos valores faltantes tem em cada uma e seguir as estratégias. Para essa variável, a melhor ideia aqui não é imputar porque queremos prever justamente ela, então excluir acaba sendo uma melhor opção.

In [12]:
percentual_nulos = df.select([
    F.round(
        F.mean(F.col(c).isNull().cast("int")) * 100,
        2
    ).alias(c)
    for c in df.columns
])

percentual_nulos.show()

+-------+--------------+--------+------+---------+-----+-----+-------------+-------------+--------------+--------------+------------+----------------+-----------------+--------------------+----------------+-----------------------+--------------------+---------------------+--------------------------------+---------------------------+---------------------+---+
|id_foco|data_hora_foco|satelite|estado|municipio|bioma|  frp|dia_sem_chuva|latitude_foco|longitude_foco|codigo_estacao|nome_estacao|latitude_estacao|longitude_estacao|distancia_estacao_km|temperatura_ar_c|umidade_relativa_ar_pct|precipitacao_inpe_mm|precipitacao_inmet_mm|pressao_atmosferica_nivel_mar_mb|vento_velocidade_horaria_ms|radiacao_global_kj_m2|ano|
+-------+--------------+--------+------+---------+-----+-----+-------------+-------------+--------------+--------------+------------+----------------+-----------------+--------------------+----------------+-----------------------+--------------------+---------------------+-----

In [13]:
df = df.filter(F.col("frp").isNotNull())

In [24]:
df.count()

4529195

Retiramos 17,68% dos registros, ficando com 4,5 milhões aproximadamente

Indo para outras colunas, as hipóteses que temos consideram principalmente umidade_relativa, precipitação_total e vento_velocidade, então as colunas que tem alto indice de nulos como "temperatura_ar", "radiacao_global" e "pressao_atmosferica_nivel_mar_mb" podemos simplesmente remover, deixando apenas a parte do vento para analise da h2 secundaria, em um modelo complementar, a justificativa de não dixar essas outras em modelo complementar vem abaixo:

In [14]:
from pyspark.sql import functions as F

cols_complementares = [
    "temperatura_ar_c",
    "radiacao_global_kj_m2",
    "pressao_atmosferica_nivel_mar_mb",
    "vento_velocidade_horaria_ms"
]

total = df.count()

completos = df.filter(
    F.col(cols_complementares[0]).isNotNull() &
    F.col(cols_complementares[1]).isNotNull() &
    F.col(cols_complementares[2]).isNotNull() &
    F.col(cols_complementares[3]).isNotNull()
).count()

print(f"Total: {total:,}")
print(f"Registros completos: {completos:,}")
print(f"Percentual: {completos / total * 100:.2f}%")

Total: 4,529,195
Registros completos: 370
Percentual: 0.01%


Percebemos que teriamos pouquissimos registros completos, então faz sentido remover elas, seguir com um modelo de 4,5 milhões sem vento, e depois pegar um modelo complementar com 630 mil linhas apenas para a hipotese do vento e comparar ambos os modelos

In [15]:
colunas_remover = [
    "temperatura_ar_c",
    "radiacao_global_kj_m2",
    "pressao_atmosferica_nivel_mar_mb"
]

df = df.drop(*colunas_remover)

In [30]:
df.printSchema()

root
 |-- id_foco: long (nullable = true)
 |-- data_hora_foco: timestamp (nullable = true)
 |-- satelite: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- bioma: string (nullable = true)
 |-- frp: double (nullable = true)
 |-- dia_sem_chuva: double (nullable = true)
 |-- latitude_foco: double (nullable = true)
 |-- longitude_foco: double (nullable = true)
 |-- codigo_estacao: string (nullable = true)
 |-- nome_estacao: string (nullable = true)
 |-- latitude_estacao: double (nullable = true)
 |-- longitude_estacao: double (nullable = true)
 |-- distancia_estacao_km: double (nullable = true)
 |-- umidade_relativa_ar_pct: double (nullable = true)
 |-- precipitacao_inpe_mm: double (nullable = true)
 |-- precipitacao_inmet_mm: double (nullable = true)
 |-- vento_velocidade_horaria_ms: double (nullable = true)
 |-- ano: integer (nullable = true)



Verificando o que faltou de nulos ainda:

In [16]:
from pyspark.sql import functions as F

total = df.count()

df.select([
    F.round(
        F.mean(F.col(c).isNull().cast("int")) * 100,
        2
    ).alias(c)
    for c in df.columns
]).show()

+-------+--------------+--------+------+---------+-----+---+-------------+-------------+--------------+--------------+------------+----------------+-----------------+--------------------+-----------------------+--------------------+---------------------+---------------------------+---+
|id_foco|data_hora_foco|satelite|estado|municipio|bioma|frp|dia_sem_chuva|latitude_foco|longitude_foco|codigo_estacao|nome_estacao|latitude_estacao|longitude_estacao|distancia_estacao_km|umidade_relativa_ar_pct|precipitacao_inpe_mm|precipitacao_inmet_mm|vento_velocidade_horaria_ms|ano|
+-------+--------------+--------+------+---------+-----+---+-------------+-------------+--------------+--------------+------------+----------------+-----------------+--------------------+-----------------------+--------------------+---------------------+---------------------------+---+
|    0.0|           0.0|     0.0|   0.0|      0.0|  0.0|0.0|         5.88|          0.0|           0.0|           0.0|         0.0|        

Vamos calcular agora a nossa variavel de precipitação real, a ideia é pegar os dados do inpe que são mais completos (3% aprox. de nulos), substituir pelos do inmet (caso tenha), e caso não tenha em ambos imputar, usando flag também

In [17]:
from pyspark.sql import functions as F

df.select(
    F.count("*").alias("total"),

    F.sum(
        F.col("precipitacao_inpe_mm").isNull().cast("int")
    ).alias("inpe_null"),

    F.sum(
        (
            F.col("precipitacao_inpe_mm").isNull() &
            F.col("precipitacao_inmet_mm").isNotNull()
        ).cast("int")
    ).alias("inpe_null_inmet_disponivel"),

    F.sum(
        (
            F.col("precipitacao_inpe_mm").isNull() &
            F.col("precipitacao_inmet_mm").isNull()
        ).cast("int")
    ).alias("ambos_null")
).show()

+-------+---------+--------------------------+----------+
|  total|inpe_null|inpe_null_inmet_disponivel|ambos_null|
+-------+---------+--------------------------+----------+
|4529195|   167328|                     98790|     68538|
+-------+---------+--------------------------+----------+



In [18]:
df_precip_comparacao = df.filter(
    F.col("precipitacao_inpe_mm").isNotNull() &
    F.col("precipitacao_inmet_mm").isNotNull()
)

df_precip_comparacao.select(
    F.corr(
        "precipitacao_inpe_mm",
        "precipitacao_inmet_mm"
    ).alias("correlacao")
).show()

+--------------------+
|          correlacao|
+--------------------+
|0.021543535235528693|
+--------------------+



In [19]:
df_precip_comparacao.select(
    F.avg(
        F.abs(
            F.col("precipitacao_inpe_mm") -
            F.col("precipitacao_inmet_mm")
        )
    ).alias("diferenca_absoluta_media")
).show()

+------------------------+
|diferenca_absoluta_media|
+------------------------+
|     0.37265392618008736|
+------------------------+



In [20]:
df.filter(
    F.col("precipitacao_inpe_mm").isNotNull() &
    F.col("precipitacao_inmet_mm").isNotNull()
).select(
    F.expr("""
        percentile_approx(
            precipitacao_inpe_mm,
            array(0.5, 0.75, 0.9, 0.95, 0.99)
        )
    """).alias("inpe"),
    
    F.expr("""
        percentile_approx(
            precipitacao_inmet_mm,
            array(0.5, 0.75, 0.9, 0.95, 0.99)
        )
    """).alias("inmet")
).show(truncate=False)

+---------------------------+-------------------------+
|inpe                       |inmet                    |
+---------------------------+-------------------------+
|[0.0, 0.0, 0.1, 1.18, 9.51]|[0.0, 0.0, 0.0, 0.0, 0.0]|
+---------------------------+-------------------------+



In [21]:
comparacao = df.filter(
    F.col("precipitacao_inpe_mm").isNotNull() &
    F.col("precipitacao_inmet_mm").isNotNull()
).withColumn(
    "chuva_inpe",
    (F.col("precipitacao_inpe_mm") > 0).cast("int")
).withColumn(
    "chuva_inmet",
    (F.col("precipitacao_inmet_mm") > 0).cast("int")
)

comparacao.groupBy(
    "chuva_inpe",
    "chuva_inmet"
).count().show()

+----------+-----------+-------+
|chuva_inpe|chuva_inmet|  count|
+----------+-----------+-------+
|         0|          1|    261|
|         1|          1|    491|
|         1|          0| 331643|
|         0|          0|2514749|
+----------+-----------+-------+



Inicialmente, pensamos em usar os dados de precipitação do INMET para completar os casos em que a precipitação do INPE estivesse nula. Porém, comparando as duas fontes, percebemos que elas apresentam uma correlação muito baixa, de aproximadamente 0,02. Além disso, dos mais de 332 mil casos em que o INPE indicou ocorrência de chuva, apenas 491 também apresentaram chuva nos dados do INMET. Por conta dessa diferença, não parece adequado misturar diretamente as duas fontes usando um coalesce

Como a precipitação do INPE possui apenas cerca de 3,7% de valores faltantes, vamos utilizá-la como nossa principal variável de precipitação e tratar os poucos valores nulos posteriormente por meio de imputação, mantendo também uma flag para indicar quais valores foram originalmente nulos.

In [22]:
# removendo do inmet

df = df.drop("precipitacao_inmet_mm")

In [23]:
# criando flag anets da imputação das variaveis

from pyspark.sql import functions as F


df = (
    df
    .withColumn(
        "umidade_missing",
        F.col("umidade_relativa_ar_pct").isNull().cast("int")
    )
    .withColumn(
        "dia_sem_chuva_missing",
        F.col("dia_sem_chuva").isNull().cast("int")
    )
    .withColumn(
        "precipitacao_missing",
        F.col("precipitacao_inpe_mm").isNull().cast("int")
    )
)

In [24]:
# calculando medianas

mediana_precip = df.approxQuantile(
    "precipitacao_inpe_mm",
    [0.5],
    0.001
)[0]

mediana_umidade = df.approxQuantile(
    "umidade_relativa_ar_pct",
    [0.5],
    0.001
)[0]

mediana_dia_sem_chuva = df.approxQuantile(
    "dia_sem_chuva",
    [0.5],
    0.001
)[0]

print("Mediana precipitação:", mediana_precip)
print("Mediana umidade:", mediana_umidade)
print("Mediana dias sem chuva:", mediana_dia_sem_chuva)

Mediana precipitação: 0.0
Mediana umidade: 36.0
Mediana dias sem chuva: 17.0


In [25]:
valores_imputacao = {
    "precipitacao_inpe_mm": mediana_precip,
    "umidade_relativa_ar_pct": mediana_umidade,
    "dia_sem_chuva": mediana_dia_sem_chuva
}

df = df.fillna(valores_imputacao)

In [28]:
from pyspark.sql.window import Window

w = Window.partitionBy("municipio").orderBy("data")

dias_municipio = (
    df_check
    .select("municipio", "data")
    .distinct()
    .withColumn(
        "data_anterior",
        F.lag("data").over(w)
    )
    .withColumn(
        "intervalo_dias",
        F.datediff("data", "data_anterior")
    )
)

dias_municipio.select(
    F.avg("intervalo_dias").alias("intervalo_medio"),
    F.expr(
        "percentile_approx(intervalo_dias, array(0.5, 0.75, 0.9, 0.95, 0.99))"
    ).alias("percentis")
).show(truncate=False)

+-----------------+-------------------+
|intervalo_medio  |percentis          |
+-----------------+-------------------+
|6.289116839090537|[1, 4, 11, 22, 105]|
+-----------------+-------------------+



Verificando mais uma vez os valores nulos percentuais em cada coluna temos:

In [29]:
from pyspark.sql import functions as F

total = df.count()

df.select([
    F.round(
        F.mean(F.col(c).isNull().cast("int")) * 100,
        2
    ).alias(c)
    for c in df.columns
]).show()

+-------+--------------+--------+------+---------+-----+---+-------------+-------------+--------------+--------------+------------+----------------+-----------------+--------------------+-----------------------+--------------------+---------------------------+---+---------------+---------------------+--------------------+
|id_foco|data_hora_foco|satelite|estado|municipio|bioma|frp|dia_sem_chuva|latitude_foco|longitude_foco|codigo_estacao|nome_estacao|latitude_estacao|longitude_estacao|distancia_estacao_km|umidade_relativa_ar_pct|precipitacao_inpe_mm|vento_velocidade_horaria_ms|ano|umidade_missing|dia_sem_chuva_missing|precipitacao_missing|
+-------+--------------+--------+------+---------+-----+---+-------------+-------------+--------------+--------------+------------+----------------+-----------------+--------------------+-----------------------+--------------------+---------------------------+---+---------------+---------------------+--------------------+
|    0.0|           0.0|    

In [30]:
df.count()

4529195

Todos tratados (com exceção de vento_velocidade_horaria_ms que iremos incluir em um modelo complementar para teste de h2), ficando com uma contagem final de 4,5 milhões de linhas para o modelo principal e aproximadamente 630 mil linhas para o modelo complementar

Vamos criar novamente a variável target, mas agora com o nome final que será `FOCO_INTENSO`:

In [32]:
mediana_frp = df.approxQuantile(
    "frp",
    [0.5],
    0.001
)[0]

df = df.withColumn(
    "FOCO_INTENSO",
    F.when(F.col("frp") > mediana_frp, 1).otherwise(0)
)

In [35]:
df.show()

+------------+-------------------+--------+----------+--------------------+--------------+-----+-------------+-------------+--------------+--------------+-------------------+----------------+-----------------+--------------------+-----------------------+--------------------+---------------------------+----+---------------+---------------------+--------------------+------------+
|     id_foco|     data_hora_foco|satelite|    estado|           municipio|         bioma|  frp|dia_sem_chuva|latitude_foco|longitude_foco|codigo_estacao|       nome_estacao|latitude_estacao|longitude_estacao|distancia_estacao_km|umidade_relativa_ar_pct|precipitacao_inpe_mm|vento_velocidade_horaria_ms| ano|umidade_missing|dia_sem_chuva_missing|precipitacao_missing|FOCO_INTENSO|
+------------+-------------------+--------+----------+--------------------+--------------+-----+-------------+-------------+--------------+--------------+-------------------+----------------+-----------------+--------------------+--------

In [37]:
df.groupBy("FOCO_INTENSO") \
    .count() \
    .withColumn(
        "percentual",
        F.round(F.col("count") / df.count() * 100, 2)
    ) \
    .show()

+------------+-------+----------+
|FOCO_INTENSO|  count|percentual|
+------------+-------+----------+
|           1|2248980|     49.66|
|           0|2280215|     50.34|
+------------+-------+----------+



Percebemos um balanço bem melhor agora entre as classes por conta da mediana

Vamos criar 5 features novas para colocar no modelo:

1. mês -> extraida de data_hora_foco, permitindo que o modelo capture sazonalidade
2. seca_prolongada -> extraida de dia_sem_chuva, verificar através dos quartis um bom ponto (variável binária), permite testar melhor as hipóteses
3. baixa_umidade -> extraida da coluna de umidade, pegar os que estão dentro do primeiro quartil (25%), mesma funcionalidade da de cima
4. condicao_seca -> junção entre seca_prolongada e baixa_umidade, junção das features, faz muito sentido para as hipóteses do que somente o mês ou dia da semana, dizendo para o modelo quando existe uma condição seca que pode ter alta correlação com focos de incêndio
5. periodo_dia -> verifica como se comporta o foco durante o dia, se é de manhã, tarde, noite ou madrugada, vem de data_hora_foco

In [38]:
# 1.mês

df = df.withColumn(
    "mes",
    F.month("data_hora_foco")
)

In [39]:
# 2.seca_prolongada

df.approxQuantile(
    "dia_sem_chuva",
    [0.25, 0.5, 0.75, 0.9, 0.95],
    0.001
)

[6.0, 17.0, 52.0, 95.0, 116.0]

Ao analisar os quartis, 50% tem valor até 17, o que seria ruim para dizer que é uma seca prolongada já que a ideia é pegar valores mais extremos, então pegar os 75% seria mais interessante nesse caso

In [ ]:
df = df.withColumn(
    "seca_prolongada",
    F.when(
        F.col("dia_sem_chuva") >= 52,
        1
    ).otherwise(0)
)

Dessa maneira conseguimos extrair informações mais interessantes no modelo

In [43]:
# 3. baixa_umidade

q25_umidade = df.approxQuantile(
    "umidade_relativa_ar_pct",
    [0.25],
    0.001
)[0]

df = df.withColumn(
    "baixa_umidade",
    F.when(
        F.col("umidade_relativa_ar_pct") <= q25_umidade,
        1
    ).otherwise(0)
)

In [44]:
# 4. condição seca

df = df.withColumn(
    "condicao_seca",
    F.when(
        (F.col("seca_prolongada") == 1) &
        (F.col("baixa_umidade") == 1),
        1
    ).otherwise(0)
)

In [47]:
# 5. periodo do dia

df = df.withColumn(
    "hora",
    F.hour("data_hora_foco")
)

df = df.withColumn(
    "periodo_dia",
    F.when(F.col("hora").between(6, 11), "manha")
     .when(F.col("hora").between(12, 17), "tarde")
     .when(F.col("hora").between(18, 23), "noite")
     .otherwise("madrugada")
)

E ficamos finalmente com esse resultado:

In [48]:
df.show()

+------------+-------------------+--------+----------+--------------------+--------------+-----+-------------+-------------+--------------+--------------+-------------------+----------------+-----------------+--------------------+-----------------------+--------------------+---------------------------+----+---------------+---------------------+--------------------+------------+---+--------------------+---------------+-------------+-------------+----+-----------+
|     id_foco|     data_hora_foco|satelite|    estado|           municipio|         bioma|  frp|dia_sem_chuva|latitude_foco|longitude_foco|codigo_estacao|       nome_estacao|latitude_estacao|longitude_estacao|distancia_estacao_km|umidade_relativa_ar_pct|precipitacao_inpe_mm|vento_velocidade_horaria_ms| ano|umidade_missing|dia_sem_chuva_missing|precipitacao_missing|FOCO_INTENSO|mes|faixa_dias_sem_chuva|seca_prolongada|baixa_umidade|condicao_seca|hora|periodo_dia|
+------------+-------------------+--------+----------+------------

Agora vamos selecionar as features:

### Meteorológicas

- **`dia_sem_chuva`**: representa a quantidade de dias consecutivos sem chuva antes da ocorrência do foco. É uma variável importante para representar a duração das condições de seca e está diretamente relacionada à hipótese principal.

- **`precipitacao_inpe_mm`**: representa a precipitação acumulada no dia até o momento da detecção do foco. Permite avaliar se menores níveis de precipitação estão relacionados a uma maior probabilidade de ocorrência de focos intensos.

- **`umidade_relativa_ar_pct`**: representa a umidade relativa do ar associada ao foco. É importante para verificar se condições de menor umidade estão relacionadas a uma maior probabilidade de `FOCO_INTENSO = 1`.

### Engenheiradas

- **`seca_prolongada`**: variável binária criada a partir de `dia_sem_chuva`. Identifica os registros com 52 dias ou mais sem chuva, valor correspondente aproximadamente ao terceiro quartil (P75) da distribuição.

- **`baixa_umidade`**: variável binária que identifica os registros pertencentes ao quartil de menor umidade relativa do ar. A ideia é representar de forma mais direta situações de baixa umidade.

- **`condicao_seca`**: variável binária que combina `seca_prolongada` e `baixa_umidade`. Recebe valor 1 quando as duas condições acontecem ao mesmo tempo, permitindo representar situações mais fortes de seca.

- **`mes`**: variável extraída da data de ocorrência do foco. Foi criada para permitir que o modelo considere possíveis efeitos de sazonalidade ao longo do ano.

- **`periodo_dia`**: variável criada a partir do horário de ocorrência do foco, dividindo os registros entre madrugada, manhã, tarde e noite. Permite verificar se o período do dia possui alguma relação com a classificação do foco.

### Contextuais

- **`bioma`**: representa o bioma onde o foco ocorreu. Diferentes biomas possuem características distintas de vegetação e condições ambientais, que podem influenciar a intensidade dos focos.

- **`estado`**: representa o estado onde o foco foi identificado. Permite considerar diferenças geográficas e climáticas existentes entre os estados analisados.

- **`distancia_estacao_km`**: representa a distância entre o foco e a estação meteorológica associada. Essa distância pode influenciar o quanto as informações meteorológicas da estação representam as condições reais no local do foco.

- **`satelite`**: identifica o satélite responsável pela detecção do foco. Diferentes satélites e sensores podem apresentar diferenças na forma como os focos são detectados, por isso essa informação pode ser relevante para o modelo.

### Flags de imputação

- **`dia_sem_chuva_missing`**: indica se o valor original de `dia_sem_chuva` estava ausente e precisou ser imputado. Permite preservar para o modelo a informação de que aquele valor não foi originalmente observado.

- **`precipitacao_missing`**: indica se o valor de precipitação do INPE estava originalmente ausente e foi preenchido durante a etapa de imputação.

- **`umidade_missing`**: indica se o valor de umidade relativa do ar estava originalmente ausente e foi preenchido durante a etapa de imputação.

In [53]:
features_candidatas = [
    # Meteorológicas
    "dia_sem_chuva",
    "precipitacao_inpe_mm",
    "umidade_relativa_ar_pct",

    # Engenheiradas
    "seca_prolongada",
    "baixa_umidade",
    "condicao_seca",
    "mes",
    "periodo_dia",

    # Contextuais
    "bioma",
    "estado",
    "distancia_estacao_km",
    "satelite",

    # Flags de imputação
    "dia_sem_chuva_missing",
    "precipitacao_missing",
    "umidade_missing"
]

df_silver = df.select(['FOCO_INTENSO', *features_candidatas])

Com isso ficamos com o seguinte dataset silver:

In [54]:
df_silver.show()

+------------+-------------+--------------------+-----------------------+---------------+-------------+-------------+---+-----------+--------------+----------+--------------------+--------+---------------------+--------------------+---------------+
|FOCO_INTENSO|dia_sem_chuva|precipitacao_inpe_mm|umidade_relativa_ar_pct|seca_prolongada|baixa_umidade|condicao_seca|mes|periodo_dia|         bioma|    estado|distancia_estacao_km|satelite|dia_sem_chuva_missing|precipitacao_missing|umidade_missing|
+------------+-------------+--------------------+-----------------------+---------------+-------------+-------------+---+-----------+--------------+----------+--------------------+--------+---------------------+--------------------+---------------+
|           0|          5.0|                 0.0|                   36.0|              0|            0|            0|  1|  madrugada|Mata Atlântica|     BAHIA|   55.46103933707039| NOAA-21|                    0|                   0|              1|
|   

Vamos salvar esse dataset particionado por ano que é a escolha mais natural a se fazer

In [55]:
df.write \
    .mode("overwrite") \
    .partitionBy("ano") \
    .parquet("../data/silver")

diario de bordo:

- precisei das 12h úteis de um dia inteiro para conseguir fazer a etapa 3 e 4, na quarta-feira anterior a entrega na quinta por conta de choques de horário
- levei entre 2 a 3 dias com menos horas somente pra juntar os dados de ambos e investigar e entender os dados